In [ ]:
import time
import heapq  

def in_mt(mt):
    res = ""
    for i in range(3):
        row = ""
        for j in range(3):
            val = mt[i*3 + j]
            if val == 0:
                row += " [ ] "
            else:
                row += f"  {val}  "
        res += row + "\n"
    res += "-" * 20 + "\n"
    return res

def get_successors_ucs(mt):
    pos = mt.index(0)
    r, c = pos // 3, pos % 3
    successors = []
    
    def swap(mt, i, j):
        new_mt = list(mt)
        new_mt[i], new_mt[j] = new_mt[j], new_mt[i]
        return new_mt, new_mt[i] # Trả về ma trận mới và giá trị của ô vừa dịch chuyển
        
    if r > 0: 
        new_state, cost = swap(mt, pos, pos - 3)
        successors.append(("Lên", new_state, cost))
    if r < 2: 
        new_state, cost = swap(mt, pos, pos + 3)
        successors.append(("Xuống", new_state, cost))
    if c > 0: 
        new_state, cost = swap(mt, pos, pos - 1)
        successors.append(("Trái", new_state, cost))
    if c < 2: 
        new_state, cost = swap(mt, pos, pos + 1)
        successors.append(("Phải", new_state, cost))
    return successors

def ucs_solve(start_state, goal_state, mode="late"):
    if start_state == goal_state:
        return [], 0
        
    # Frontier lưu trữ dưới dạng danh sách (heap): 
    # Mỗi phần tử là tuple: (tổng_chi_phí_g, thứ_tự_tăng_dần_chống_trùng, node_hiện_tại, đường_đi)
    # Lưu ý: Thêm `counter` để tránh lỗi so sánh ma trận trực tiếp khi chi phí g bằng nhau
    counter = 0
    frontier = []
    heapq.heappush(frontier, (0, counter, start_state, []))
    
    # Tập hợp quản lý chi phí tối ưu nhất từng tìm thấy cho mỗi trạng thái
    explored = {tuple(start_state): 0}
    nodes_generated = 1
    
    while frontier:
        # Lấy trạng thái có tổng chi phí g nhỏ nhất ra duyệt
        g, _, node, path = heapq.heappop(frontier)
        
        # Nếu chạy mode "late" (Kiểm tra đích khi lấy ra khỏi Frontier - Chuẩn UCS)
        if mode == "late" and node == goal_state:
            return path, nodes_generated
            
        # Nếu trạng thái lấy ra có chi phí lớn hơn chi phí tối ưu đã lưu trước đó -> Bỏ qua
        if mode == "late" and g > explored.get(tuple(node), float('inf')):
            continue
            
        for action, child, step_cost in get_successors_ucs(node):
            new_g = g + step_cost # Chi phí bước này bằng chính số trên ô vừa di chuyển
            child_tuple = tuple(child)
            
            if mode == "early":
                nodes_generated += 1
                # Kiểm tra đích ngay khi sinh node (Sẽ chạy nhanh hơn nhưng không đảm bảo đường đi có chi phí rẻ nhất)
                if child == goal_state:
                    return path + [(action, child)], nodes_generated
                    
                if child_tuple not in explored or new_g < explored[child_tuple]:
                    explored[child_tuple] = new_g
                    counter += 1
                    heapq.heappush(frontier, (new_g, counter, child, path + [(action, child)]))
            else:
                # Mode "late": Cập nhật Frontier nếu tìm thấy chi phí tốt hơn
                if child_tuple not in explored or new_g < explored[child_tuple]:
                    explored[child_tuple] = new_g
                    nodes_generated += 1
                    counter += 1
                    heapq.heappush(frontier, (new_g, counter, child, path + [(action, child)]))
                    
    return None, nodes_generated